<a href="https://colab.research.google.com/github/dalmasm/AI/blob/Data-Science/AI_Engineer_2025_Transformers_Semantic_Search_and_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Engineer 2025 - Semantic Search,RAG

## Objetivo
Mostrar las bases de implementacion de Semantic Search y RAG y comparar con busqueda tradicional por keyword

### Detalles
- Este notebook muestra como realizar busquedas contextuales contra una base de conocimiento simulada, usando la base como conocimiento factico y un modelo AI para generar la respuesta contextual

### Referencia
Ver clase 45

### Consultas
albertcapacitaciondigital@gmail.com
    

In [8]:
!pip uninstall faiss-cpu numpy -y

Found existing installation: faiss-cpu 1.8.0.post1
Uninstalling faiss-cpu-1.8.0.post1:
  Successfully uninstalled faiss-cpu-1.8.0.post1
Found existing installation: numpy 2.3.2
Uninstalling numpy-2.3.2:
  Successfully uninstalled numpy-2.3.2


In [9]:
#Instalar FAISS (busqueda en vectores), Numpy (manejo de vectores en Python)
!pip install -U faiss-cpu numpy==1.24.4 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 82.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray 2025.7.1 requires numpy>=1.26, but you have numpy 1.24.4 which is incompatible.
arviz 0.22.0 requires numpy>=1.26.0, but you have numpy 1.24.4 which is incompatible.
jaxlib 0.5.3 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.24.4 which is incompatible.
blosc2 3.6.1 requires numpy>=1.26, but you have numpy 1.24.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.24.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_ver

In [1]:
#Test para chequear compatibilidad de versiones entre FAISS y Numpy
#No todas las versiones de numpy son compatibles!
# CUIDADO! Si este paso da error en Colab, reiniciar la Session desde el menu
# Runtime Restart Session y probar nuevamente
import numpy as np
import faiss

# Crear un vector al azar para simular embedding
embeds_test = np.random.rand(15, 4096).astype(np.float32)

# Crear FAISS index
dim_test = embeds_test.shape[1]
index_test = faiss.IndexFlatL2(dim_test)
index_test.add(embeds_test)

#Esto deberia dar OK
print("FAISS index works. Vectors indexed:", index_test.ntotal)

FAISS index works. Vectors indexed: 15


In [2]:
# %%capture
#Instalar Cohere (para conectarse con la API, obtener embeddings)
#Rankbm 25 (search), Sentence Transformers
!pip install cohere==5.5.8 rank_bm25==0.2.2 sentence-transformers==3.0.1




  Using cached numpy-2.3.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
Using cached numpy-2.3.2-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
faiss-cpu 1.8.0.post1 requires numpy<2.0,>=1.0, but you have numpy 2.3.2 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.2 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.2 which is incompatible.
opencv-python 4.12.0.88 requ

# Dense Retrieval


## 1. Crear un documento simulado y transformarlo en embeddings


In [3]:
# Usar Cohere como servicio de embeddings
import cohere

# No me afanen la key, saquen la suya en el sitio de Cohere
# https://cohere.com/
api_key = "hklm4scijVqyQLtvTJvnH19h66FKMvuqe8LiYiMi"

# Conectarse con Cohere
co = cohere.Client(api_key)

In [4]:
#Documento de prueba, simulado como una gran variable string
#Reemplaza una knowledge base, forma parte del "corpus" de conocimiento del
#modelo
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

# Estrategia de chunking, separar por oraciones
texts = text.split('.')

# Eliminar saltos de linea ("enters" en el texto)
texts = [t.strip(' \n') for t in texts]

## 2. Obtener Embeddings de Cohere


In [5]:
import numpy as np

# Pasarle a Cohere el documento en chunks
response = co.embed(
  texts=texts,
  input_type="search_document",
).embeddings

#La respuesta es un vector, usamos un array de Numpy como equivalente
embeds = np.array(response)
print(embeds.shape)

(15, 4096)


## 3. Creando un index de Search con FAISS


In [7]:
#FAISS permite indexar vectores para realizar busquedas rapidas sobre los mismos
import faiss

#Dimension de embeddings
dim = embeds.shape[1]
#crear un indice en FAISS con esa dimension
index = faiss.IndexFlatL2(dim)
#agregar al indice los embeddings que obtuvimos en el paso anterior
index.add(np.float32(embeds))

## 4. Semantic Search

In [8]:
import pandas as pd

# Funcion de busqueda
def search(query, number_of_results=3):

  # 1. Transformar el query en embeddings con Cohere
  query_embed = co.embed(texts=[query],
                input_type="search_query",).embeddings[0]

  # 2. Usar el indice para ver que embeddings son "mas cercanos"/relevantes
  distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

  # 3. Formatear el resultado como Dataframe
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                              'distance': distances[0]})

  # 4. Imprimir resultado
  print(f"Query:'{query}'\nNearest neighbors:")
  return results

In [9]:
#Testeando con un query de ejemplo
query = "how precise was the science"
results = search(query)
results

TypeError: Cannot convert numpy.ndarray to numpy.ndarray

## 3. Comparando con Lexical Search (BM25)


In [ ]:
#Importar rankbm25 (lexical search)
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

# Tokenizador para normalizar texto y eliminar palabras vacías y puntuación
def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

In [ ]:
from tqdm import tqdm

# Preprocesamos todo el corpus para BM25
# Usando el mismo texto original
tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████| 15/15 [00:00<00:00, 47411.12it/s]


In [ ]:
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    # Paso 1: puntuaciones de BM25
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    # Paso 2: obtener los mejores candidatos
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    # Paso 3: mostrar resultados
    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))


In [ ]:
#Lexical Search, ejemplo
keyword_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


## Limitaciones de Dense Retrieval


In [ ]:
#Enviamos un query que pregunta por info que NO esta en el texto
# Esto causa una alucinacion tecnica al haber una respuesta siempre
# No importa la distancia entre vectores
# En una aplicacion real se puede setear un maximo de distancia para mostrar
# resultados
# Otra limitacion obvia es que no es una busqueda por keyword match
# Si las palabras no existen en el texto, igual habra un match
query = "What is the mass of the moon?"
results = search(query)
results

Query:'What is the mass of the moon?'
Nearest neighbors:


,texts,distance
0,Cinematographer Hoyte van Hoytema shot it on 3...,12854.443359
1,The film had a worldwide gross over $677 milli...,13301.007812
2,It has also received praise from many astronom...,13332.000977


# Reranking con Cohere


In [ ]:
#Re ranking permite re ordenar resultados iniciales de busqueda
#Para poner mas arriba los mas relevantes
# Le enviamos el query, mas los resultados iniciales y obtenemos un nuevo
# orden de search results
query = "how precise was the science"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15239799),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.050354082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424)]

In [ ]:
#Mostrar resultados en orden de relevancia
for idx, result in enumerate(results.results):
    print(idx, result.relevance_score , result.document.text)

0 0.15239799 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 0.050354082 The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2 0.0350424 Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


# Reranking y Lexical Search combinados


In [ ]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    #Agregar re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))

In [ ]:
#Hybrid Search
keyword_and_reranking_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.032	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine
	0.031	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of In

# Retrieval-Augmented Generation

## Grounded Generation (RAG)


In [ ]:
query = "income generated"

# Embedding Search
results = search(query)

# Grounded Generation
# Usando los resultados de la busqueda, utilizar un modelo AI para
# generar la respuesta con el contexto correcto
docs_dict = [{'text': text} for text in results['texts']]
response = co.chat(
    message = query,
    documents=docs_dict
)

print(response.text)

Query:'income generated'
Nearest neighbors:
The film Interstellar generated a worldwide gross of over $677 million, and $773 million with subsequent re-releases, making it the tenth-highest grossing film of 2014.


In [ ]:
response

NonStreamedChatResponse(text='The film Interstellar generated a worldwide gross of over $677 million, and $773 million with subsequent re-releases, making it the tenth-highest grossing film of 2014.', generation_id='462bf1fb-cc82-435b-91d2-939b58e3e042', citations=[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_1', 'doc_2'], type='TEXT_CONTENT'), ChatCitation(start=34, end=70, text='worldwide gross of over $677 million', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=76, end=116, text='$773 million with subsequent re-releases', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=132, end=168, text='tenth-highest grossing film of 2014.', document_ids=['doc_0'], type='TEXT_CONTENT')], documents=[{'id': 'doc_1', 'text': 'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar'}, {'id': 'doc_2', 'text'

In [ ]:
# Fuentes usadas por el modelo
response.citations

[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_1', 'doc_2'], type='TEXT_CONTENT'),
 ChatCitation(start=34, end=70, text='worldwide gross of over $677 million', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=76, end=116, text='$773 million with subsequent re-releases', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=132, end=168, text='tenth-highest grossing film of 2014.', document_ids=['doc_0'], type='TEXT_CONTENT')]